# Multi-source candidate dataset (Kaggle, CPU)

Attach your CSV as a Kaggle dataset; enable Internet and the `GEE_JSON_KEY`,
`PLANET_API_KEY`, and `HF_TOKEN` secrets. Start with `DRY_RUN=True`.

This standalone notebook exports Global Quarterly Planet imagery, Sentinel-1 VV/VH,
slope, and aspect on per-candidate UTM grids. A 1,000 m footprint uses 333
pixels at approximately 3.003 m, preserving the exact footprint. Resampling
does not increase native sensor resolution.

**NDVI is a screening heuristic, not evidence that a tile is landslide-free.**
Original labels are preserved; shifted dates and fallback outcomes are explicit.
Planet visual RGB/RGBA is never used for NDVI. Separate Sentinel-2 surface-
reflectance B4 (red) and B8 (NIR) seasonal composites provide screening at 10 m.
Quarterly mosaics can contain pixels from multiple dates. Original-date
fallback does not certify a negative label either.

Only complete six-layer candidates enter verified archives. Failed rows remain
in global metadata/status tables. Credentials and signed URLs are not exported.
Cloud authentication, catalogue availability, and imagery must be tested in
Kaggle with your own permissions.

## 1. Dependencies and configuration
Install only missing packages. All tunables are in the next CONFIG cell.

In [ ]:
# cell: dependencies
import importlib.util
import subprocess
import sys

DEPENDENCIES = {
    "ee": "earthengine-api", "requests": "requests", "rasterio": "rasterio",
    "pyproj": "pyproj", "pandas": "pandas", "numpy": "numpy", "scipy": "scipy",
    "huggingface_hub": "huggingface_hub", "tqdm": "tqdm", "shapely": "shapely",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import calendar
import contextlib
import hashlib
import json
import logging
import math
import os
import re
import shutil
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from copy import deepcopy
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from threading import local
from urllib.parse import urljoin, urlsplit

import ee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from huggingface_hub import CommitOperationAdd, HfApi
from huggingface_hub.errors import HfHubHTTPError
from pyproj import CRS, Transformer
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.vrt import WarpedVRT
from scipy import ndimage
from shapely.geometry import Polygon
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
LOG = logging.getLogger("candidate_dataset")
for name in ("urllib3", "httpx", "huggingface_hub", "rasterio"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

In [ ]:
# cell: config
CONFIG = {
    "INPUT_CSV": "/kaggle/input/YOUR_DATASET/non_landslide.csv",
    "OUTPUT_DIR": "/kaggle/working/multisource_candidates_quarterly_s2",
    "RANDOM_SEED": 42,
    "YEAR_SHIFT_FRACTION": 0.30,
    "TARGET_YEAR": 2018,
    "YEAR_SHIFT_ONLY_LABEL": None,
    "DRY_SEASON_MONTHS": [10, 11, 12],
    "BOX_SIZE_M": 1000.0,
    "OUTPUT_PIXEL_SIZE_M": 3.0,
    "RETRY_COUNT": 4,
    "RETRY_BASE_SECONDS": 2,
    "HTTP_TIMEOUT_SECONDS": 180,
    "HF_REPO_ID": "sasudo2/landslide-multisource-candidates",
    "HF_PRIVATE": True,
    "HF_PATH_IN_REPO": "multisource_candidates",
    "DRY_RUN": True,
    "DRY_RUN_COUNT": 3,
    "PLANET_SERIES_ID": None,
    "PLANET_MOSAIC_NAME_TEMPLATE": "global_quarterly_{year}q{quarter}_mosaic",
    "PLANET_BAND_METADATA": None,
    "PLANET_BAND_METADATA_SOURCE": None,
    "S2_COLLECTION": "COPERNICUS/S2_SR_HARMONIZED",
    "S2_PIXEL_SIZE_M": 10.0,
    "S2_MAX_SCENE_CLOUD_PERCENT": 80,
    "S2_CLEAR_SCL_CLASSES": [4, 5, 6, 7],
    "DEM_ASSET": "COPERNICUS/DEM/GLO30",
    "DEM_IS_COLLECTION": True,
    "DEM_BAND": "DEM",
    "DEM_NATIVE_SCALE_M": 30,
    "S1_COLLECTION": "COPERNICUS/S1_GRD",
    "NDVI_DROP_THRESHOLD": 0.25,
    "MAX_LOSS_FRACTION": 0.02,
    "MAX_LOSS_PATCH_AREA_M2": 1500.0,
    "MIN_VALID_FRACTION": 0.70,
    "MEAN_NDVI_DROP_THRESHOLD": None,
    "REFLECTANCE_MIN": 0.0,
    "REFLECTANCE_MAX": 1.2,
    "CALIBRATE_NDVI": False,
    "CALIBRATION_LANDSLIDE_CSV": None,
    "CALIBRATION_MAX_SAMPLES": 30,
    "BATCH_SIZE": 50,
    "MAX_WORKERS": 4,
    "MIN_FREE_GB": 2.0,
    "MAX_BATCH_UNCOMPRESSED_GB": 1.0,
    "DOWNLOAD_WORKSPACE_GB": 0.5,
    "DELETE_AFTER_VERIFIED_UPLOAD": True,
}
# Global Quarterly is auto-selected by exact series name; an explicit series ID
# must also refer to Global Quarterly. TIFF RGB labels take precedence over the
# documented visual-product RGB/RGBA band order. Overrides require a source.
# Sentinel-2 SR coverage in 2017-2018 is not global; missing/insufficient data
# triggers original-date fallback, never a silently substituted reflectance type.

## 2. Safe IO, retries, and authentication
Validate all three credentials even in dry-run mode. A valid Planet API key
still requires Global Quarterly basemap download entitlement over Nepal.
Do not enable HTTP debug logs. No exception includes a raw request URL.

In [ ]:
# cell: io
BASEMAPS_URL = "https://api.planet.com/basemaps/v1/"
LAYERS = ("planet_before", "planet_after", "slope", "aspect", "s1_before", "s1_after")
WORKER_LOCAL = local()


def planet_session():
    if not getattr(WORKER_LOCAL, "active", False):
        return PLANET
    if getattr(WORKER_LOCAL, "planet", None) is None:
        session = requests.Session()
        session.auth = PLANET.auth
        session.headers.update(PLANET.headers)
        WORKER_LOCAL.planet = session
    return WORKER_LOCAL.planet


def process_candidate_worker(record):
    WORKER_LOCAL.active = True
    try:
        return process_record(deepcopy(record))
    finally:
        session = getattr(WORKER_LOCAL, "planet", None)
        if session is not None:
            session.close()
            del WORKER_LOCAL.planet
        WORKER_LOCAL.active = False


class MissingLayer(RuntimeError):
    pass


def error_summary(exc):
    response = getattr(exc, "response", None)
    status = getattr(response, "status_code", None)
    if isinstance(exc, (ValueError, MissingLayer, FileNotFoundError)):
        return f"{type(exc).__name__}: {exc}"
    return f"{type(exc).__name__}" + (f" (HTTP {status})" if status else "")


def retry_call(operation, label):
    for attempt in range(CONFIG["RETRY_COUNT"]):
        try:
            return operation()
        except (requests.RequestException, HfHubHTTPError, ee.EEException, OSError) as exc:
            status = getattr(getattr(exc, "response", None), "status_code", None)
            if status and status not in (408, 429) and status < 500:
                raise RuntimeError(f"{label}: HTTP {status}; check permissions/input") from None
            if attempt + 1 == CONFIG["RETRY_COUNT"]:
                raise RuntimeError(f"{label}: retries exhausted ({error_summary(exc)})") from None
            LOG.warning("%s: retry %d/%d (%s)", label, attempt + 1,
                        CONFIG["RETRY_COUNT"], type(exc).__name__)
            time.sleep(CONFIG["RETRY_BASE_SECONDS"] * 2 ** attempt)


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))
    os.replace(temporary, path)


def disk_free(path):
    return shutil.disk_usage(Path(path)).free


def require_space(path, additional_bytes=0):
    if disk_free(path) < CONFIG["MIN_FREE_GB"] * 1024 ** 3 + additional_bytes:
        raise OSError("Insufficient free disk space; upload/clean a verified batch or reduce batch size")


def http_json(session, url, params=None):
    def operation():
        with session.get(url, params=params, timeout=CONFIG["HTTP_TIMEOUT_SECONDS"]) as response:
            response.raise_for_status()
            return response.json()
    return retry_call(operation, "API request")


def planet_pages(session, url, key, params=None):
    seen = set()
    while url:
        url = urljoin(BASEMAPS_URL, url)
        if urlsplit(url).hostname != "api.planet.com":
            raise ValueError("Unexpected catalogue pagination host")
        if url in seen:
            raise ValueError("Repeated catalogue pagination link")
        seen.add(url)
        payload = http_json(session, url, params)
        yield from payload.get(key, [])
        url = payload.get("_links", {}).get("_next")
        params = None


def download_file(session, url, output):
    output = Path(output)
    partial = output.with_suffix(output.suffix + ".part")
    def operation():
        partial.unlink(missing_ok=True)
        require_space(output.parent)
        with session.get(url, stream=True, timeout=CONFIG["HTTP_TIMEOUT_SECONDS"]) as response:
            response.raise_for_status()
            require_space(output.parent, int(response.headers.get("Content-Length", 0)))
            with partial.open("wb") as stream:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        require_space(output.parent, len(chunk))
                        stream.write(chunk)
        os.replace(partial, output)
    try:
        retry_call(operation, "Raster download")
    finally:
        partial.unlink(missing_ok=True)


def validate_config():
    if (isinstance(CONFIG["MAX_WORKERS"], bool)
            or not isinstance(CONFIG["MAX_WORKERS"], int) or CONFIG["MAX_WORKERS"] < 1):
        raise ValueError("MAX_WORKERS must be a positive integer (use 1 for sequential processing)")
    if not 0 <= CONFIG["YEAR_SHIFT_FRACTION"] <= 1:
        raise ValueError("YEAR_SHIFT_FRACTION must lie in [0, 1]")
    if CONFIG["BOX_SIZE_M"] <= 0 or CONFIG["OUTPUT_PIXEL_SIZE_M"] <= 0:
        raise ValueError("Footprint and pixel sizes must be positive")
    if round(CONFIG["BOX_SIZE_M"] / CONFIG["OUTPUT_PIXEL_SIZE_M"]) < 1:
        raise ValueError("Requested pixel size gives an empty grid")
    months = CONFIG["DRY_SEASON_MONTHS"]
    if len(months) != 3 or any(m not in range(1, 13) for m in months):
        raise ValueError("DRY_SEASON_MONTHS must be three consecutive calendar months")
    if any(months[i] != (months[0] - 1 + i) % 12 + 1 for i in range(3)):
        raise ValueError("Dry-season months must be consecutive and ordered")
    if months[0] not in (1, 4, 7, 10):
        raise ValueError("Global Quarterly requires calendar-quarter dry-season months")
    for key in ("MAX_LOSS_FRACTION", "MIN_VALID_FRACTION"):
        if not 0 <= CONFIG[key] <= 1:
            raise ValueError(f"{key} must lie in [0, 1]")
    if min(CONFIG["RETRY_COUNT"], CONFIG["BATCH_SIZE"], CONFIG["CALIBRATION_MAX_SAMPLES"]) < 1:
        raise ValueError("Retry count, batch size, and calibration sample count must be positive")
    if not 0 < CONFIG["NDVI_DROP_THRESHOLD"] <= 2 or CONFIG["MAX_LOSS_PATCH_AREA_M2"] <= 0:
        raise ValueError("NDVI drop must be in (0, 2] and loss-patch area must be positive")
    if CONFIG["S2_PIXEL_SIZE_M"] < 10:
        raise ValueError("Sentinel-2 NDVI screening grid must be at least 10 m")
    if math.floor(CONFIG["BOX_SIZE_M"] / CONFIG["S2_PIXEL_SIZE_M"]) < 1:
        raise ValueError("Sentinel-2 pixel size gives an empty screening grid")
    if not CONFIG["S2_CLEAR_SCL_CLASSES"] or any(
        value not in (4, 5, 6, 7) for value in CONFIG["S2_CLEAR_SCL_CLASSES"]
    ):
        raise ValueError("S2_CLEAR_SCL_CLASSES must be a non-empty subset of [4, 5, 6, 7]")
    if not 0 <= CONFIG["S2_MAX_SCENE_CLOUD_PERCENT"] <= 100:
        raise ValueError("S2_MAX_SCENE_CLOUD_PERCENT must lie in [0, 100]")
    if any(CONFIG[key] < 0 for key in ("MIN_FREE_GB", "DOWNLOAD_WORKSPACE_GB")):
        raise ValueError("Disk reserve values must be non-negative")
    if CONFIG["MAX_BATCH_UNCOMPRESSED_GB"] <= 0:
        raise ValueError("MAX_BATCH_UNCOMPRESSED_GB must be positive")
    if not 1 <= CONFIG["TARGET_YEAR"] <= 9999:
        raise ValueError("TARGET_YEAR is outside the supported calendar")
    template = CONFIG["PLANET_MOSAIC_NAME_TEMPLATE"]
    if template:
        try:
            template.format(year=2018, month=10, quarter=4)
        except (KeyError, ValueError, IndexError):
            raise ValueError("Invalid quarterly mosaic name template; use {year}, {quarter}") from None
    prefix = CONFIG["HF_PATH_IN_REPO"].strip("/")
    if not prefix or ".." in Path(prefix).parts:
        raise ValueError("HF_PATH_IN_REPO must be a dedicated relative directory")
    if not re.fullmatch(r"[^/\s]+/[^/\s]+", CONFIG["HF_REPO_ID"]):
        raise ValueError("HF_REPO_ID must be owner/dataset")


def authenticate():
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    values = {}
    for name in ("GEE_JSON_KEY", "PLANET_API_KEY", "HF_TOKEN"):
        try:
            value = secrets.get_secret(name)
        except Exception:
            raise RuntimeError(f"Missing/inaccessible Kaggle secret: {name}") from None
        if not value or not value.strip():
            raise RuntimeError(f"Empty Kaggle secret: {name}")
        values[name] = value.strip()
    try:
        key = json.loads(values["GEE_JSON_KEY"])
        if not all(key.get(k) for k in ("client_email", "private_key", "project_id")):
            raise ValueError("Incomplete service-account JSON")
        with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as handle:
            key_path = Path(handle.name)
            os.chmod(key_path, 0o600)
            handle.write(values["GEE_JSON_KEY"])
        try:
            credentials = ee.ServiceAccountCredentials(key["client_email"], str(key_path))
            ee.Initialize(credentials, project=key["project_id"])
            retry_call(lambda: ee.Number(1).getInfo(), "GEE authentication")
        finally:
            key_path.unlink(missing_ok=True)
    except Exception:
        raise RuntimeError("GEE_JSON_KEY authentication failed; check JSON, project registration and access") from None
    planet = requests.Session()
    planet.auth = (values["PLANET_API_KEY"], "")
    try:
        series = list(planet_pages(planet, BASEMAPS_URL + "series/", "series"))
    except Exception:
        raise RuntimeError("PLANET_API_KEY validation failed; check key and Basemaps entitlement") from None
    hf = HfApi(token=values["HF_TOKEN"])
    try:
        retry_call(hf.whoami, "Hugging Face authentication")
    except Exception:
        raise RuntimeError("HF_TOKEN validation failed; check token validity and dataset write scope") from None
    values.clear()
    LOG.info("GEE, Planet and Hugging Face authentication validated")
    return planet, hf, series

## 3. Input validation, exact metric footprints, year shifting, and seasons
Actual CSV names (`incident`, `candidate`, `min_lon`, etc.) are supported.
Duplicate candidate keys are all rejected; invalid rows are logged separately.
PRE/POST windows are calendar-date inclusive in metadata and end-exclusive for
API queries. Windows strictly exclude the incident date, even on boundaries.
Year-shift sampling operates on candidate pairs, not incident_id alone.

In [ ]:
# cell: input_grid
def read_candidates(csv_path):
    frame = pd.read_csv(csv_path, dtype=str)
    print("CSV columns:", frame.columns.tolist())
    aliases = {
        "incident_id": ("incident_id", "incident"),
        "candidate_id": ("candidate_id", "candidate"),
        "west": ("west", "min_lon"), "east": ("east", "max_lon"),
        "south": ("south", "min_lat"), "north": ("north", "max_lat"),
    }
    for canonical, options in aliases.items():
        present = [name for name in options if name in frame.columns]
        if len(present) != 1:
            raise ValueError(f"Expected exactly one column for {canonical}: {options}")
        frame = frame.rename(columns={present[0]: canonical})
    required = ("incident_id", "candidate_id", "incident_date", "west", "south", "east", "north")
    if any(column not in frame for column in required):
        raise ValueError(f"CSV requires {required}")
    errors = pd.Series("", index=frame.index)
    for column in ("incident_id", "candidate_id"):
        frame[column] = frame[column].str.strip()
        invalid = ~frame[column].str.fullmatch(r"[A-Za-z0-9-]+", na=False)
        errors.loc[invalid] += f"invalid {column}; "
    duplicate = frame.duplicated(["incident_id", "candidate_id"], keep=False)
    errors.loc[duplicate] += "duplicate incident/candidate pair; "
    dates = pd.to_datetime(frame["incident_date"], format="mixed", errors="coerce", utc=True)
    errors.loc[dates.isna()] += "invalid incident_date; "
    for column in ("west", "south", "east", "north"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    bounds = frame[["west", "south", "east", "north"]]
    invalid = (
        ~np.isfinite(bounds).all(axis=1)
        | ~frame.west.between(-180, 180) | ~frame.east.between(-180, 180)
        | ~frame.south.between(-90, 90) | ~frame.north.between(-90, 90)
        | (frame.west >= frame.east) | (frame.south >= frame.north)
        | ~((frame.south + frame.north) / 2).between(-80, 84)
    )
    errors.loc[invalid] += "invalid bbox or outside UTM latitude coverage; "
    rejected = frame.loc[errors.ne("")].copy()
    rejected["validation_error"] = errors[errors.ne("")]
    for index, reason in errors[errors.ne("")].items():
        LOG.warning("Skipping CSV row %d: %s", index + 2, reason)
    frame = frame.loc[errors.eq("")].copy()
    frame["original_incident_date"] = dates.loc[frame.index].dt.strftime("%Y-%m-%d")
    if "label" not in frame:
        LOG.warning("No label column: NDVI calibration unavailable without a separate labeled CSV")
    frame = frame.sort_values(["incident_id", "candidate_id"]).reset_index(drop=True)
    frame["key"] = frame.incident_id + "_" + frame.candidate_id
    frame = frame.astype(object).where(frame.notna(), None)
    return frame, rejected


def grid_for_row(row):
    lon = (float(row["west"]) + float(row["east"])) / 2
    lat = (float(row["south"]) + float(row["north"])) / 2
    zone = min(60, max(1, math.floor((lon + 180) / 6) + 1))
    epsg = (32600 if lat >= 0 else 32700) + zone
    forward = Transformer.from_crs(4326, epsg, always_xy=True)
    backward = Transformer.from_crs(epsg, 4326, always_xy=True)
    cx, cy = forward.transform(lon, lat)
    half = CONFIG["BOX_SIZE_M"] / 2
    bounds = (cx - half, cy - half, cx + half, cy + half)
    shape = round(CONFIG["BOX_SIZE_M"] / CONFIG["OUTPUT_PIXEL_SIZE_M"])
    transform = from_bounds(*bounds, shape, shape)
    corners = [backward.transform(x, y) for x, y in (
        (bounds[0], bounds[1]), (bounds[2], bounds[1]),
        (bounds[2], bounds[3]), (bounds[0], bounds[3]), (bounds[0], bounds[1]),
    )]
    if max(x for x, _ in corners) - min(x for x, _ in corners) > 180:
        raise ValueError("Footprint crosses the antimeridian; split it before processing")
    original_xy = [forward.transform(x, y) for x, y in (
        (float(row["west"]), float(row["south"])),
        (float(row["east"]), float(row["south"])),
        (float(row["east"]), float(row["north"])),
        (float(row["west"]), float(row["north"])),
    )]
    xs, ys = zip(*original_xy)
    return {
        "epsg": epsg, "bounds_utm": bounds, "corners_wgs84": corners,
        "bbox_wgs84": Polygon(corners).bounds, "width": shape, "height": shape,
        "transform": tuple(transform)[:6], "pixel_size_m": transform.a,
        "original_width_m": max(xs) - min(xs), "original_height_m": max(ys) - min(ys),
    }


def shift_candidates(frame):
    frame = frame.copy()
    frame["shifted_incident_date"] = frame["original_incident_date"]
    frame["year_shifted"] = False
    eligible = pd.Series(True, index=frame.index)
    label = CONFIG["YEAR_SHIFT_ONLY_LABEL"]
    if label is not None:
        if "label" not in frame:
            raise ValueError("YEAR_SHIFT_ONLY_LABEL requires a label column")
        eligible &= frame.label.str.lower().eq(str(label).lower())
    if label is None:
        LOG.warning("Year shifting applies to all labels: real-landslide pre/post context may become wrong")
    count = int(round(eligible.sum() * CONFIG["YEAR_SHIFT_FRACTION"]))
    chosen = np.random.default_rng(CONFIG["RANDOM_SEED"]).choice(
        frame.index[eligible].to_numpy(), size=count, replace=False,
    )
    for index in chosen:
        original = date.fromisoformat(frame.at[index, "original_incident_date"])
        day = min(original.day, calendar.monthrange(CONFIG["TARGET_YEAR"], original.month)[1])
        shifted = date(CONFIG["TARGET_YEAR"], original.month, day)
        if shifted != original:
            frame.at[index, "shifted_incident_date"] = shifted.isoformat()
            frame.at[index, "year_shifted"] = True
    frame["date_used"] = np.where(frame.year_shifted, "shifted", "original")
    frame["effective_incident_date"] = frame["shifted_incident_date"]
    return frame


def season_windows(incident_date):
    incident = date.fromisoformat(str(incident_date)[:10])
    start_month = CONFIG["DRY_SEASON_MONTHS"][0]
    windows = []
    for year in range(incident.year - 3, incident.year + 4):
        start = date(year, start_month, 1)
        month_index = year * 12 + start_month - 1 + 3
        end_exclusive = date(month_index // 12, month_index % 12 + 1, 1)
        windows.append((start, end_exclusive))
    pre = max((w for w in windows if w[1] - timedelta(days=1) < incident), key=lambda w: w[0])
    post = min((w for w in windows if w[0] > incident), key=lambda w: w[0])
    return {side: {"start": start.isoformat(), "end_exclusive": end.isoformat(),
                   "end_inclusive": (end - timedelta(days=1)).isoformat()}
            for side, (start, end) in zip(("before", "after"), (pre, post))}


def select_run_rows(frame):
    if not CONFIG["DRY_RUN"]:
        return frame.copy()
    count = CONFIG["DRY_RUN_COUNT"]
    if count != 3 or len(frame) < count or not frame.year_shifted.any():
        raise ValueError("Dry-run requires three valid rows and at least one actual year-shifted row")
    selected = [frame.index[frame.year_shifted][0]]
    selected += [i for i in frame.index if i not in selected][:count - 1]
    return frame.loc[selected].reset_index(drop=True)

## 4. Global Quarterly Planet imagery (not the NDVI source)
Select the exact named Global Quarterly mosaic for each PRE/POST window.
RGB/RGBA visual imagery is accepted for dataset inputs, not for NDVI.
Band labels and product type are recorded. The documented RGB/RGBA order is
used only for byte Global Quarterly products without unambiguous TIFF labels.
Alpha is validity, never NIR. All delivered bands and source sample values are
retained on the requested output grid; float32 storage does not add radiometry.
Quads are downloaded one at a time, warped, and removed. No nearby-quarter
substitution is made because that would change the pre/post interpretation.

In [ ]:
# cell: raster_planet
def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def raster_path(folder, row, layer):
    return Path(folder) / f"{row['key']}_{layer}.tif"


def raster_info(path, grid, fingerprint=None, expected_count=None):
    with rasterio.open(path) as src:
        expected = rasterio.Affine(*grid["transform"])
        if (src.crs != CRS.from_epsg(grid["epsg"]) or src.width != grid["width"]
                or src.height != grid["height"] or not src.transform.almost_equals(expected, precision=1e-7)):
            raise ValueError(f"Grid mismatch: {Path(path).name}")
        if expected_count is not None and src.count != expected_count:
            raise ValueError(f"Band-count mismatch: {Path(path).name}")
        tags = src.tags()
        if fingerprint is not None and tags.get("request_fingerprint") != fingerprint:
            raise ValueError(f"Stale raster provenance: {Path(path).name}")
        metadata = json.loads(tags.get("layer_metadata", "{}"))
        data = src.read(masked=True).astype(np.float32).filled(np.nan)
        usable = np.all(np.isfinite(data), axis=0)
        names = metadata.get("band_order", [])
        if "alpha" in names:
            usable &= data[names.index("alpha")] > 0
        if not usable.any():
            raise ValueError(f"No valid data: {Path(path).name}")
        return metadata


def cached_raster(path, grid, fingerprint, expected_count=None):
    if not Path(path).exists():
        return None
    try:
        return raster_info(path, grid, fingerprint, expected_count)
    except (ValueError, rasterio.errors.RasterioError) as exc:
        LOG.warning("Rebuilding cached %s: %s", Path(path).name, error_summary(exc))
        return None


def write_grid(path, data, grid, fingerprint, metadata, descriptions):
    require_space(Path(path).parent, data.nbytes * 2)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp.tif")
    try:
        with rasterio.open(
            temporary, "w", driver="GTiff", width=grid["width"], height=grid["height"],
            count=data.shape[0], dtype="float32", crs=f"EPSG:{grid['epsg']}",
            transform=rasterio.Affine(*grid["transform"]), nodata=np.nan,
            compress="deflate",
        ) as dst:
            dst.write(data.astype(np.float32))
            dst.descriptions = tuple(descriptions)
            dst.update_tags(request_fingerprint=fingerprint,
                            layer_metadata=json.dumps(metadata, allow_nan=False))
        raster_info(temporary, grid, fingerprint, data.shape[0])
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def normal_band_name(value):
    value = re.sub(r"[^a-z0-9]", "", str(value).lower())
    return {
        "b": "blue", "blue": "blue", "g": "green", "green": "green",
        "r": "red", "red": "red", "n": "nir", "nir": "nir",
        "nearinfrared": "nir", "nearinfrared1": "nir", "alpha": "alpha",
    }.get(value, value)


def planet_band_metadata(src, mosaic):
    def usable(names):
        names = [normal_band_name(name) for name in names]
        return (names if len(names) == src.count and
                all(names.count(name) == 1 for name in ("red", "blue", "green")) else None)
    descriptions = [
        src.descriptions[i - 1] or src.tags(i).get("NAME")
        or src.tags(i).get("BAND_NAME") or interpretation.name
        for i, interpretation in enumerate(src.colorinterp, 1)
    ]
    names = usable(descriptions)
    origin = "TIFF band descriptions/tags/color interpretation"
    if names is None:
        bands = mosaic.get("bands", [])
        if isinstance(bands, list):
            names = usable([band.get("name", band.get("id", "")) if isinstance(band, dict)
                            else band for band in bands])
        origin = "Planet mosaic API band metadata"
    if names is None and CONFIG["PLANET_BAND_METADATA"] is not None:
        if not CONFIG["PLANET_BAND_METADATA_SOURCE"]:
            raise ValueError("Explicit band metadata requires PLANET_BAND_METADATA_SOURCE")
        names = usable(CONFIG["PLANET_BAND_METADATA"])
        origin = "verified override: " + CONFIG["PLANET_BAND_METADATA_SOURCE"]
    visual = all(dtype == "uint8" for dtype in src.dtypes)
    if names is None and visual and src.count in (3, 4) and str(
        mosaic.get("name", "")
    ).startswith("global_quarterly_"):
        names = ["red", "green", "blue"] + (["alpha"] if src.count == 4 else [])
        origin = "Global Quarterly visual RGB/RGBA product specification: https://docs.planet.com/data/imagery/mosaics/"
        LOG.warning("Using documented Global Quarterly visual band order: %s", names)
    if names is None:
        raise ValueError("Missing RGB band labels; configure verified "
                         "PLANET_BAND_METADATA and PLANET_BAND_METADATA_SOURCE")
    return {
        "band_order": names, "band_metadata_source": origin,
        "source_scales": list(src.scales), "source_offsets": list(src.offsets),
        "source_dtypes": list(src.dtypes),
        "product_type": "visual" if visual else "non-byte Global Quarterly imagery",
        "used_for_ndvi": False, "mask_source": "GeoTIFF nodata/alpha",
        "native_pixel_size": list(src.res),
        "native_crs": src.crs.to_string() if src.crs else None,
    }


def choose_planet_series(series):
    print("Available Planet series (no credentials):")
    for item in series:
        print(" ", item.get("id"), item.get("name"))
    selected_id = CONFIG["PLANET_SERIES_ID"]
    if selected_id:
        selected = [s for s in series if s.get("id") == selected_id]
    else:
        selected = [
            s for s in series
            if str(s.get("name", "")).strip().lower() == "global quarterly"
        ]
    if len(selected) != 1:
        raise ValueError("Set PLANET_SERIES_ID to the entitled Global Quarterly series above")
    if str(selected[0].get("name", "")).strip().lower() != "global quarterly":
        raise ValueError("Selected series must be Global Quarterly; NDVI uses separate Sentinel-2 imagery")
    return selected[0]


def nominal_mosaic_window(mosaic):
    name = str(mosaic.get("name", ""))
    match = re.search(r"(20\d{2})[-_]?q([1-4])(?!\d)", name, re.I)
    if match:
        year, quarter = map(int, match.groups())
        month = (quarter - 1) * 3 + 1
        start = date(year, month, 1)
        index = year * 12 + month - 1 + 3
        end = date(index // 12, index % 12 + 1, 1)
        return start.isoformat(), end.isoformat()
    return None


def find_mosaic(catalogue, window):
    start = date.fromisoformat(window["start"])
    template = CONFIG["PLANET_MOSAIC_NAME_TEMPLATE"]
    if template:
        exact_name = template.format(year=start.year, month=start.month,
                                     quarter=(start.month - 1) // 3 + 1)
        matches = [m for m in catalogue if m.get("name") == exact_name]
    else:
        matches = [m for m in catalogue if nominal_mosaic_window(m) ==
                   (window["start"], window["end_exclusive"])]
    if not matches:
        raise MissingLayer(f"No exact Global Quarterly mosaic for {window['start']} "
                           f"through {window['end_inclusive']}")
    if len(matches) != 1:
        raise ValueError("Multiple mosaics match one quarter; configure an exact mosaic name template")
    mosaic = matches[0]
    if nominal_mosaic_window(mosaic) != (window["start"], window["end_exclusive"]):
        raise ValueError("Selected mosaic does not match the exact seasonal quarter")
    return mosaic


def pull_planet(row, grid, folder, layer, window, fingerprint):
    path = raster_path(folder, row, layer)
    cached = cached_raster(path, grid, fingerprint)
    if cached is not None:
        return cached
    mosaic = find_mosaic(MOSAIC_CATALOGUE, window)
    details_url = BASEMAPS_URL + f"mosaics/{mosaic['id']}/"
    session = planet_session()
    details = http_json(session, details_url)
    mosaic = {**mosaic, **details}
    quad_url = details_url + "quads/"
    quads = list(planet_pages(
        session, quad_url, "items",
        {"bbox": ",".join(map(str, grid["bbox_wgs84"])), "_page_size": 50},
    ))
    if not quads:
        raise MissingLayer("No Planet quads cover this footprint")
    result, metadata, descriptions = None, None, None
    with tempfile.TemporaryDirectory(dir=ROOT / ".scratch", prefix="planet_") as temporary:
        quad_path = Path(temporary) / "quad.tif"
        for quad in quads:
            link = quad.get("_links", {}).get("download")
            if not link:
                raise MissingLayer("Planet quad has no permitted download link")
            # Do not forward Basic Auth to a signed object-storage host.
            with requests.Session() as anonymous:
                download_session = session if urlsplit(link).hostname == "api.planet.com" else anonymous
                download_file(download_session, link, quad_path)
            with rasterio.open(quad_path) as src:
                if src.crs is None:
                    raise ValueError("Planet quad has no CRS")
                current = planet_band_metadata(src, mosaic)
                if metadata is not None and any(current[k] != metadata[k] for k in (
                    "band_order", "source_scales", "source_offsets", "source_dtypes",
                )):
                    raise ValueError("Planet quad spectral metadata disagree")
                metadata = current
                descriptions = current["band_order"]
                with WarpedVRT(
                    src, crs=f"EPSG:{grid['epsg']}", transform=rasterio.Affine(*grid["transform"]),
                    width=grid["width"], height=grid["height"], dtype="float32",
                    nodata=np.nan, resampling=Resampling.nearest,
                ) as vrt:
                    data = vrt.read(masked=True).astype(np.float32).filled(np.nan)
                usable = np.all(np.isfinite(data), axis=0)
                if "alpha" in descriptions:
                    usable &= data[descriptions.index("alpha")] > 0
                data[:, ~usable] = np.nan
                if result is None:
                    result = data
                else:
                    replace = ~np.all(np.isfinite(result), axis=0) & usable
                    result[:, replace] = data[:, replace]
            quad_path.unlink()
    metadata.update(
        mosaic_id=mosaic["id"], mosaic_name=mosaic["name"], series_id=PLANET_SERIES["id"],
        nominal_window=window, quad_count=len(quads), resampling="nearest",
        first_acquired=mosaic.get("first_acquired"), last_acquired=mosaic.get("last_acquired"),
        datatype=mosaic.get("datatype"), ndvi_source=CONFIG["S2_COLLECTION"],
    )
    write_grid(path, result, grid, fingerprint, metadata, descriptions)
    return metadata


def report_quarterly_coverage(records, catalogue):
    requested = {}
    for row in records:
        dates = {row["original_incident_date"], row["shifted_incident_date"]}
        for incident_date in dates:
            for window in season_windows(incident_date).values():
                requested[window["start"]] = window
    missing = []
    for start, window in sorted(requested.items()):
        try:
            find_mosaic(catalogue, window)
        except MissingLayer:
            missing.append(start[:7])
    print(f"Required nominal quarterly periods: {len(requested)}; missing: {len(missing)}")
    if missing:
        LOG.warning("No accessible Global Quarterly mosaic for: %s. Affected seasons are incomplete; "
                    "shifted seasons trigger original-date fallback.",
                    ", ".join(missing))
    return missing

## 5. GEE terrain and Sentinel-1 support layers
Copernicus GLO30 is an ImageCollection. Terrain is calculated on a projected
30 m DEM before resampling to the output grid, with a buffer for derivatives.
Aspect is nearest-neighbour resampled to avoid averaging 359 and 1 degrees.
Sentinel-1 GRD is **already in dB**; median composites are kept in dB and are
not log-transformed again. One orbit direction is chosen from the total PRE
and POST scene counts, deterministically preferring ascending on a tie.

In [ ]:
# cell: gee_layers
def gee_geometry(grid):
    return ee.Geometry.Polygon([grid["corners_wgs84"]], proj="EPSG:4326", geodesic=False)


def pull_gee(image, row, grid, folder, layer, fingerprint, metadata, bands):
    path = raster_path(folder, row, layer)
    cached = cached_raster(path, grid, fingerprint, len(bands))
    if cached is not None:
        return cached
    params = {
        "format": "GEO_TIFF", "crs": f"EPSG:{grid['epsg']}",
        "crs_transform": list(grid["transform"]),
        "dimensions": [grid["width"], grid["height"]],
        "filePerBand": False,
    }
    image = image.select(bands).toFloat().unmask(-9999, sameFootprint=False)
    with tempfile.TemporaryDirectory(dir=ROOT / ".scratch", prefix="gee_") as temporary:
        raw = Path(temporary) / "download.tif"
        def request_and_download():
            url = image.getDownloadURL(params)
            with requests.Session() as session:
                download_file(session, url, raw)
        retry_call(request_and_download, "GEE grid export")
        with rasterio.open(raw) as src:
            # Validate the requested server grid before adding local provenance.
            if (src.crs != CRS.from_epsg(grid["epsg"]) or
                    (src.width, src.height) != (grid["width"], grid["height"]) or
                    not src.transform.almost_equals(rasterio.Affine(*grid["transform"]), precision=1e-7)):
                raise ValueError(f"GEE export grid mismatch for {layer}")
            data = src.read().astype(np.float32)
            data[data == -9999] = np.nan
    write_grid(path, data, grid, fingerprint, metadata, bands)
    return metadata


def terrain_images(grid):
    aoi = gee_geometry(grid).buffer(120)
    if CONFIG["DEM_IS_COLLECTION"]:
        dem = ee.ImageCollection(CONFIG["DEM_ASSET"]).filterBounds(aoi).select(CONFIG["DEM_BAND"]).mosaic()
    else:
        dem = ee.Image(CONFIG["DEM_ASSET"]).select(CONFIG["DEM_BAND"])
    dem = dem.reproject(crs=f"EPSG:{grid['epsg']}", scale=CONFIG["DEM_NATIVE_SCALE_M"])
    terrain = ee.Terrain.products(dem)
    return {layer: terrain.select(layer) for layer in ("slope", "aspect")}


def s2_screening_grid(grid):
    shape = math.floor(CONFIG["BOX_SIZE_M"] / CONFIG["S2_PIXEL_SIZE_M"])
    transform = from_bounds(*grid["bounds_utm"], shape, shape)
    return {**grid, "width": shape, "height": shape,
            "transform": tuple(transform)[:6], "pixel_size_m": transform.a}


def mask_s2_scene(image):
    scl = image.select("SCL")
    classes = CONFIG["S2_CLEAR_SCL_CLASSES"]
    clear = scl.remap(classes, [1] * len(classes), 0).eq(1)
    spectral = image.select(["B4", "B8"], ["red", "nir"])
    valid = spectral.mask().reduce(ee.Reducer.min())
    return spectral.updateMask(clear.And(valid)).multiply(0.0001).toFloat()


def pull_s2_screening(row, grid, folder, side, window, windows):
    layer = f"s2_screen_{side}"
    fingerprint = layer_fingerprint(row, grid, layer, windows)
    path = raster_path(folder, row, layer)
    cached = cached_raster(path, grid, fingerprint, 2)
    if cached is not None:
        return cached
    collection = (
        ee.ImageCollection(CONFIG["S2_COLLECTION"]).filterBounds(gee_geometry(grid))
        .filterDate(window["start"], window["end_exclusive"])
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", CONFIG["S2_MAX_SCENE_CLOUD_PERCENT"]))
    )
    count = int(retry_call(collection.size().getInfo, "Sentinel-2 scene count"))
    if count == 0:
        raise MissingLayer(f"No Sentinel-2 SR scenes for NDVI {side} window "
                           f"{window['start']} through {window['end_inclusive']}")
    scene_ids = retry_call(
        lambda: collection.aggregate_array("system:index").getInfo(), "Sentinel-2 scene provenance",
    )
    image = collection.map(mask_s2_scene).median()
    metadata = {
        "collection": CONFIG["S2_COLLECTION"], "ndvi_source": "Sentinel-2 surface reflectance",
        "band_order": ["red", "nir"], "source_bands": ["B4", "B8"],
        "reflectance_scales": [1.0, 1.0], "reflectance_offsets": [0.0, 0.0],
        "source_reflectance_scale": 0.0001, "units": "reflectance", "used_for_ndvi": True,
        "scene_count": count, "scene_ids": scene_ids, "window": window,
        "mask_source": "Sentinel-2 SCL clear classes and joint B4/B8 masks before median",
        "clear_scl_classes": CONFIG["S2_CLEAR_SCL_CLASSES"],
        "composite_method": "per-band median of cloud-masked B4/B8 reflectance",
        "native_spectral_resolution_m": 10, "scl_resolution_m": 20,
        "screening_pixel_size_m": grid["pixel_size_m"], "resampling": "nearest",
    }
    return pull_gee(image, row, grid, folder, layer, fingerprint, metadata, ["red", "nir"])


def s1_collections(grid, windows):
    base = (
        ee.ImageCollection(CONFIG["S1_COLLECTION"]).filterBounds(gee_geometry(grid))
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    )
    collections, counts = {}, {}
    for direction in ("ASCENDING", "DESCENDING"):
        collections[direction], counts[direction] = {}, {}
        for side, window in windows.items():
            collection = base.filterDate(window["start"], window["end_exclusive"]).filter(
                ee.Filter.eq("orbitProperties_pass", direction)
            )
            collections[direction][side] = collection
            counts[direction][side] = int(retry_call(collection.size().getInfo, "S1 scene count"))
    chosen = max(counts, key=lambda direction: sum(counts[direction].values()))
    return chosen, collections[chosen], counts[chosen]


def layer_fingerprint(row, grid, layer, windows):
    return stable_hash({
        "run_signature": RUN_SIGNATURE, "key": row["key"], "grid": grid, "layer": layer,
        "windows": None if layer in ("slope", "aspect") else windows,
    })


def pull_all_layers(row, grid, folder, incident_date, terrain_cache=None):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    windows = season_windows(incident_date)
    result = {"windows": windows, "layers": {}, "layer_metadata": {}, "errors": {}}
    for side in ("before", "after"):
        layer = f"planet_{side}"
        try:
            result["layer_metadata"][layer] = pull_planet(
                row, grid, folder, layer, windows[side],
                layer_fingerprint(row, grid, layer, windows),
            )
            result["layers"][layer] = "ok"
        except Exception as exc:
            result["layers"][layer] = "missing" if isinstance(exc, MissingLayer) else "failed"
            result["errors"][layer] = error_summary(exc)
            LOG.error("%s %s: %s", row["key"], layer, error_summary(exc))
    terrain = None
    for layer in ("slope", "aspect"):
        fingerprint = layer_fingerprint(row, grid, layer, windows)
        path = raster_path(folder, row, layer)
        try:
            cached = cached_raster(path, grid, fingerprint, 1)
            if cached is None and terrain_cache is not None:
                old = raster_path(terrain_cache, row, layer)
                cached = cached_raster(old, grid, fingerprint, 1)
                if cached is not None:
                    shutil.copy2(old, path)
            if cached is None:
                if terrain is None:
                    terrain = terrain_images(grid)
                cached = pull_gee(
                    terrain[layer], row, grid, folder, layer, fingerprint,
                    {"dem_asset": CONFIG["DEM_ASSET"], "units": "degrees",
                     "terrain_scale_m": CONFIG["DEM_NATIVE_SCALE_M"], "resampling": "nearest"},
                    [layer],
                )
            result["layer_metadata"][layer] = cached
            result["layers"][layer] = "ok"
        except Exception as exc:
            result["layers"][layer] = "failed"
            result["errors"][layer] = error_summary(exc)
            LOG.error("%s %s: %s", row["key"], layer, error_summary(exc))
    try:
        direction, collections, counts = s1_collections(grid, windows)
        result["s1_orbit_direction"] = direction
        for side in ("before", "after"):
            layer = f"s1_{side}"
            try:
                if counts[side] == 0:
                    raise MissingLayer(f"No VV/VH IW scenes in {direction} {side} window")
                image = collections[side].select(["VV", "VH"]).median()
                metadata = {"orbit_direction": direction, "scene_count": counts[side],
                            "band_order": ["VV", "VH"], "units": "dB",
                            "collection": CONFIG["S1_COLLECTION"], "window": windows[side]}
                result["layer_metadata"][layer] = pull_gee(
                    image, row, grid, folder, layer,
                    layer_fingerprint(row, grid, layer, windows), metadata, ["VV", "VH"],
                )
                result["layers"][layer] = "ok"
            except Exception as exc:
                result["layers"][layer] = "missing" if isinstance(exc, MissingLayer) else "failed"
                result["errors"][layer] = error_summary(exc)
                LOG.error("%s %s: %s", row["key"], layer, error_summary(exc))
    except Exception as exc:
        for layer in ("s1_before", "s1_after"):
            result["layers"][layer] = "failed"
            result["errors"][layer] = error_summary(exc)
        LOG.error("%s Sentinel-1 selection: %s", row["key"], error_summary(exc))
    return result

## 6. NDVI screening and calibration
NDVI is computed only from separate Sentinel-2 SR B4/B8 composites, never
Planet RGB or alpha. Screening is performed on a separate 10 m UTM grid over
the same footprint and exact windows; output-grid upsampling is not used to
count patches. Source scenes are SCL-masked before the reflectance median.
Statistics use pixels valid in both dates, with configurable reflectance bounds.
SCL is 20 m and is imperfect; NDVI loss remains non-specific to landslides.
Patch areas use the actual screening pixel area (default 100 m2) and
8-connectivity. At defaults, a patch must exceed 15 pixels to exceed 1,500 m2.
Screening Red/NIR TIFFs and scene IDs are retained when successfully downloaded.
Missing Sentinel-2 SR (including limited 2017-2018 coverage) is inconclusive,
triggers shifted-date fallback, and never substitutes TOA data silently.

In [ ]:
# cell: ndvi
def ndvi_from_raster(path):
    with rasterio.open(path) as src:
        metadata = json.loads(src.tags()["layer_metadata"])
        if metadata.get("used_for_ndvi") is False:
            raise ValueError("Planet dataset imagery is not the NDVI source; use Sentinel-2 Red/NIR")
        names = metadata["band_order"]
        bands = {}
        valid = np.ones((src.height, src.width), dtype=bool)
        for name in ("red", "nir"):
            index = names.index(name)
            raw = src.read(index + 1, masked=True).astype(np.float32)
            values = raw.filled(np.nan)
            values = values * metadata["reflectance_scales"][index] + metadata["reflectance_offsets"][index]
            bands[name] = values
            valid &= np.isfinite(values) & ~np.ma.getmaskarray(raw)
            valid &= (values >= CONFIG["REFLECTANCE_MIN"]) & (values <= CONFIG["REFLECTANCE_MAX"])
        denominator = bands["nir"] + bands["red"]
        ndvi = np.full(denominator.shape, np.nan, dtype=np.float32)
        np.divide(bands["nir"] - bands["red"], denominator, out=ndvi,
                  where=np.isfinite(denominator) & (np.abs(denominator) > 1e-8))
        valid &= np.isfinite(ndvi)
        return ndvi, valid, metadata.get("mask_source", "nodata and finite reflectance bounds")


def ndvi_statistics(before, after, before_valid, after_valid, pixel_size_m):
    if not (before.shape == after.shape == before_valid.shape == after_valid.shape):
        raise ValueError("NDVI arrays/masks have different shapes")
    valid = before_valid & after_valid & np.isfinite(before) & np.isfinite(after)
    fraction = float(valid.mean())
    difference = after - before
    loss = valid & (difference < -CONFIG["NDVI_DROP_THRESHOLD"])
    labels, _ = ndimage.label(loss, structure=np.ones((3, 3), dtype=np.uint8))
    sizes = np.bincount(labels.ravel())[1:]
    largest = int(sizes.max()) if sizes.size else 0
    count = int(valid.sum())
    loss_fraction = float(loss.sum() / count) if count else None
    mean = float(difference[valid].mean()) if count else None
    max_pixels = CONFIG["MAX_LOSS_PATCH_AREA_M2"] / pixel_size_m ** 2
    stats = {
        "ndvi_loss_fraction": loss_fraction, "largest_loss_patch_px": largest,
        "largest_loss_patch_m2": largest * pixel_size_m ** 2,
        "max_loss_patch_pixels": max_pixels, "valid_fraction": fraction, "mean_dndvi": mean,
    }
    reasons = []
    if fraction < CONFIG["MIN_VALID_FRACTION"] or count == 0:
        outcome = "inconclusive"
        reasons.append("insufficient jointly valid pixels")
    else:
        if loss_fraction > CONFIG["MAX_LOSS_FRACTION"]:
            reasons.append("loss fraction exceeds threshold")
        if largest > max_pixels:
            reasons.append("connected loss patch exceeds area threshold")
        threshold = CONFIG["MEAN_NDVI_DROP_THRESHOLD"]
        if threshold is not None and mean < -threshold:
            reasons.append("mean NDVI drop exceeds threshold")
        outcome = "fail" if reasons else "pass"
    return {**stats, "ndvi_check_result": outcome, "ndvi_check_reason": "; ".join(reasons)}


def check_pair(row, grid, folder, result):
    source = {
        "ndvi_source": CONFIG["S2_COLLECTION"], "ndvi_source_bands": ["B4", "B8"],
        "ndvi_resolution_m": s2_screening_grid(grid)["pixel_size_m"],
    }
    if any(result["layers"].get(f"planet_{side}") != "ok" for side in ("before", "after")):
        missing = any(result["layers"].get(f"planet_{side}") == "missing" for side in ("before", "after"))
        return {**source,
            "ndvi_loss_fraction": None, "largest_loss_patch_px": None,
            "valid_fraction": 0.0, "mean_dndvi": None,
            "ndvi_check_result": "missing_mosaic" if missing else "inconclusive",
            "ndvi_check_reason": "Planet pair missing or failed; see layer errors",
        }
    screening_grid = s2_screening_grid(grid)
    result["screening_grid"] = screening_grid
    result["screening_layers"], result["screening_metadata"], result["screening_errors"] = {}, {}, {}
    for side in ("before", "after"):
        layer = f"s2_screen_{side}"
        try:
            metadata = pull_s2_screening(
                row, screening_grid, folder, side, result["windows"][side], result["windows"],
            )
            result["screening_layers"][layer] = "ok"
            result["screening_metadata"][layer] = metadata
        except Exception as exc:
            result["screening_layers"][layer] = "missing" if isinstance(exc, MissingLayer) else "failed"
            result["screening_errors"][layer] = error_summary(exc)
            raster_path(folder, row, layer).unlink(missing_ok=True)
            LOG.error("%s %s: %s", row["key"], layer, error_summary(exc))
    if any(value != "ok" for value in result["screening_layers"].values()):
        return {**source, "ndvi_loss_fraction": None, "largest_loss_patch_px": None,
                "valid_fraction": 0.0, "mean_dndvi": None, "ndvi_check_result": "inconclusive",
                "ndvi_check_reason": "Sentinel-2 screening pair missing/failed; see screening_errors"}
    before, before_valid, before_mask = ndvi_from_raster(raster_path(folder, row, "s2_screen_before"))
    after, after_valid, after_mask = ndvi_from_raster(raster_path(folder, row, "s2_screen_after"))
    return {
        **source,
        **ndvi_statistics(before, after, before_valid, after_valid, screening_grid["pixel_size_m"]),
        "before_mask_method": before_mask, "after_mask_method": after_mask,
    }


def remove_child_tree(path, parent):
    path, parent = Path(path).resolve(), Path(parent).resolve()
    if path.parent != parent or path == parent:
        raise ValueError("Refusing deletion outside the expected direct child directory")
    if path.exists():
        shutil.rmtree(path)


def calibrate_ndvi(frame):
    if not CONFIG["CALIBRATE_NDVI"]:
        return
    source = frame
    if CONFIG["CALIBRATION_LANDSLIDE_CSV"]:
        source, _ = read_candidates(CONFIG["CALIBRATION_LANDSLIDE_CSV"])
    if "label" not in source:
        LOG.warning("Skipping calibration: no label column")
        return
    years = pd.to_datetime(source.original_incident_date).dt.year
    positives = source.loc[source.label.str.lower().eq("landslide") & years.between(2017, 2019)]
    positives = positives.sample(
        n=min(len(positives), CONFIG["CALIBRATION_MAX_SAMPLES"]), random_state=CONFIG["RANDOM_SEED"],
    )
    negatives = frame.loc[frame.year_shifted]
    if "label" in negatives:
        negatives = negatives.loc[negatives.label.str.lower().isin(
            ["not_landslide", "non-landslide", "non_landslide"],
        )]
    negatives = negatives.head(CONFIG["CALIBRATION_MAX_SAMPLES"])
    if positives.empty:
        LOG.warning("No labeled 2017-2019 landslides for calibration; "
                    "attach a positive CSV using CALIBRATION_LANDSLIDE_CSV")
    rows = []
    for group, samples in (("known_landslide", positives), ("shifted_non_landslide", negatives)):
        for row in samples.to_dict(orient="records"):
            grid = grid_for_row(row)
            date_value = (row["original_incident_date"] if group == "known_landslide"
                          else row["shifted_incident_date"])
            folder = ROOT / ".calibration" / row["key"]
            folder.mkdir(parents=True, exist_ok=True)
            windows = season_windows(date_value)
            result = {"layers": {}, "errors": {}, "windows": windows}
            try:
                for side in ("before", "after"):
                    layer = f"planet_{side}"
                    try:
                        pull_planet(row, grid, folder, layer, windows[side],
                                    layer_fingerprint(row, grid, layer, windows))
                        result["layers"][layer] = "ok"
                    except Exception as exc:
                        result["layers"][layer] = "missing" if isinstance(exc, MissingLayer) else "failed"
                        LOG.error("Calibration %s %s: %s", row["key"], layer, error_summary(exc))
                rows.append({"key": row["key"], "group": group,
                             **check_pair(row, grid, folder, result)})
            finally:
                remove_child_tree(folder, ROOT / ".calibration")
    table = pd.DataFrame(rows)
    table.to_csv(ROOT / "ndvi_calibration.csv", index=False)
    if table.empty:
        LOG.warning("Calibration produced no samples")
        return
    print(table.to_string(index=False))
    known = table.loc[table.group.eq("known_landslide")]
    evaluable = known.loc[known.ndvi_check_result.isin(["pass", "fail"])]
    if len(known):
        print("Known landslides triggering fallback (including inconclusive/missing):",
              float(known.ndvi_check_result.ne("pass").mean()))
    if len(evaluable):
        print("Vegetation-loss rejection among evaluable known landslides:",
              float(evaluable.ndvi_check_result.eq("fail").mean()))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for axis, field in zip(axes, ("ndvi_loss_fraction", "largest_loss_patch_px")):
        for group in table.group.unique():
            values = table.loc[table.group.eq(group), field].dropna()
            if len(values):
                axis.hist(values, bins=15, alpha=0.5, label=group)
        axis.set_title(field)
        if axis.has_data():
            axis.legend()
    fig.tight_layout()
    fig.savefig(ROOT / "ndvi_calibration.png")
    plt.show()
    raise RuntimeError("Calibration finished. Review results, tune thresholds, then use a NEW "
                       "OUTPUT_DIR with CALIBRATE_NDVI=False for the full/dry run")

## 7. Persistent metadata, resumability, and transactional date fallback
A run signature prevents mixing different inputs/configurations. MAX_WORKERS
is operational only and may be changed without invalidating existing outputs. To change
thresholds or switches after a run, select a new output directory.
Staged shifted and original attempts are isolated; terrain files are reused.
Only a fully validated six-layer folder is published. A recoverable backup
protects an interrupted directory swap. NDVI failure triggers fallback, but
the original-date NDVI result is logged without rejecting the original pair.

In [ ]:
# cell: state_processing
def flatten_record(record):
    flat = {}
    def visit(prefix, value):
        if isinstance(value, dict):
            for name, child in value.items():
                visit(f"{prefix}_{name}" if prefix else name, child)
        elif isinstance(value, (tuple, list)):
            flat[prefix] = json.dumps(value)
        else:
            flat[prefix] = value
    visit("", record)
    return flat


def save_tables(records=None, folder=None):
    records = list(STATE["records"].values()) if records is None else records
    folder = ROOT if folder is None else Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    metadata = pd.DataFrame([flatten_record(record) for record in records])
    fields = ["key", "incident_id", "candidate_id", "status", "date_used",
              "year_shifted", "effective_incident_date", "archive"]
    fields += [f"layers_{layer}" for layer in LAYERS]
    fields += ["ndvi_check_result", "ndvi_check_reason", "processing_error"]
    for name, frame in (
        ("metadata.csv", metadata),
        ("status.csv", metadata.reindex(columns=fields)),
    ):
        temporary = folder / f"{name}.tmp"
        frame.to_csv(temporary, index=False)
        os.replace(temporary, folder / name)


def checkpoint():
    atomic_json(ROOT / "run_state.json", STATE)
    save_tables()


def prepare_run():
    validate_config()
    root = Path(CONFIG["OUTPUT_DIR"]).resolve()
    root.mkdir(parents=True, exist_ok=True)
    for name in (".scratch", ".staging", ".backup", ".calibration", "candidates", "archives"):
        (root / name).mkdir(exist_ok=True)
    frame, rejected = read_candidates(CONFIG["INPUT_CSV"])
    rejected.to_csv(root / "invalid_rows.csv", index=False)
    if frame.empty:
        raise ValueError("No valid candidate rows")
    shifted = shift_candidates(frame)
    run_rows = select_run_rows(shifted)
    signature = stable_hash({
        "pipeline_version": 3,
        "config": {key: value for key, value in CONFIG.items() if key != "MAX_WORKERS"},
        "input_sha256": hashlib.sha256(Path(CONFIG["INPUT_CSV"]).read_bytes()).hexdigest(),
        "calibration_sha256": (
            hashlib.sha256(Path(CONFIG["CALIBRATION_LANDSLIDE_CSV"]).read_bytes()).hexdigest()
            if CONFIG["CALIBRATION_LANDSLIDE_CSV"] else None
        ),
    })
    state_path = root / "run_state.json"
    if state_path.exists():
        state = json.loads(state_path.read_text())
        if state["signature"] != signature:
            raise ValueError("Output directory belongs to another configuration/input; use a new OUTPUT_DIR")
    else:
        records = {}
        for row in run_rows.to_dict(orient="records"):
            grid = grid_for_row(row)
            records[row["key"]] = {
                **row, "grid": grid, "status": "pending", "archive": None,
                "layers": {layer: "pending" for layer in LAYERS},
                "windows": season_windows(row["effective_incident_date"]),
                "ndvi_check_result": None, "ndvi_check_reason": "",
            }
        state = {"signature": signature, "records": records, "archives": [],
                 "created_at": datetime.now(timezone.utc).isoformat()}
    return root, shifted, state, signature


def validate_folder(folder, record):
    for layer in LAYERS:
        count = 2 if layer.startswith("s1_") else 1 if layer in ("slope", "aspect") else None
        raster_info(
            raster_path(folder, record, layer), record["grid"],
            layer_fingerprint(record, record["grid"], layer, record["windows"]), count,
        )
    for layer, status in record.get("screening_layers", {}).items():
        if status == "ok":
            grid = record["screening_grid"]
            raster_info(
                raster_path(folder, record, layer), grid,
                layer_fingerprint(record, grid, layer, record["windows"]), 2,
            )


def publish_folder(staged, destination):
    destination = Path(destination)
    backup = ROOT / ".backup" / destination.name
    if backup.exists():
        raise RuntimeError("Unrecovered publish backup exists")
    if destination.exists():
        os.replace(destination, backup)
    try:
        os.replace(staged, destination)
    except OSError:
        if backup.exists():
            os.replace(backup, destination)
        raise
    if backup.exists():
        remove_child_tree(backup, ROOT / ".backup")


def recover_record(record):
    key = record["key"]
    final = ROOT / "candidates" / key
    backup = ROOT / ".backup" / key
    if backup.exists() and not final.exists():
        os.replace(backup, final)
        LOG.warning("Recovered interrupted folder swap for %s", key)
    if final.exists() and (final / "provenance.json").exists():
        recovered = json.loads((final / "provenance.json").read_text())
        if recovered.get("run_signature") != RUN_SIGNATURE:
            raise ValueError("Published candidate has an incompatible run signature")
        try:
            validate_folder(final, recovered)
        except (ValueError, rasterio.errors.RasterioError) as exc:
            LOG.warning("Published %s needs rebuilding: %s", key, error_summary(exc))
            return None
        if backup.exists():
            remove_child_tree(backup, ROOT / ".backup")
        return recovered
    return None


def process_record(record):
    record = dict(record)
    grid, key = record["grid"], record["key"]
    recovered = recover_record(record)
    if recovered is not None:
        return recovered
    for field in ("screening_layers", "screening_grid", "screening_metadata", "screening_errors",
                  "original_ndvi", "shifted_attempt", "fallback_triggered"):
        record.pop(field, None)
    require_space(ROOT, int(CONFIG["DOWNLOAD_WORKSPACE_GB"] * 1024 ** 3))
    stage_root = ROOT / ".staging" / key
    active_kind = "shifted" if record["year_shifted"] else "original"
    active_date = record["shifted_incident_date"] if record["year_shifted"] else record["original_incident_date"]
    stage = stage_root / active_kind
    attempt = pull_all_layers(record, grid, stage, active_date)
    if record["year_shifted"]:
        stats = check_pair(record, grid, stage, attempt)
        record.update(stats)
        record["fallback_triggered"] = stats["ndvi_check_result"] != "pass"
        if record["fallback_triggered"]:
            LOG.warning("%s: shifted NDVI %s; staging original-date fallback", key, stats["ndvi_check_result"])
            shifted_stage = stage
            stage = stage_root / "original"
            original = record["original_incident_date"]
            record["shifted_attempt"] = attempt
            attempt = pull_all_layers(record, grid, stage, original, terrain_cache=shifted_stage)
            record["original_ndvi"] = check_pair(record, grid, stage, attempt)
            active_date, active_kind = original, "original"
    record.update(attempt)
    record["date_used"] = active_kind
    record["effective_incident_date"] = active_date
    record["run_signature"] = RUN_SIGNATURE
    if all(record["layers"].get(layer) == "ok" for layer in LAYERS):
        validate_folder(stage, record)
        record["status"] = "complete"
        record["processing_error"] = ""
        atomic_json(stage / "provenance.json", record)
        publish_folder(stage, ROOT / "candidates" / key)
        if stage_root.exists():
            remove_child_tree(stage_root, ROOT / ".staging")
    else:
        record["status"] = "incomplete"
        record["processing_error"] = "One or more required layers are unavailable; staged files retained for resume"
    return record

## 8. Verified ZIP batches and Hugging Face upload
Each ZIP contains candidate folders, its matching metadata/status snapshots,
and a README. Global CSVs retain all candidates, including incomplete ones.
Archive CRCs, file membership, and complete-candidate counts are checked.
Large batches are split to fit available disk space before ZIP creation.
A committed remote file's size is checked before any local candidate cleanup.

Uploads are isolated under `HF_PATH_IN_REPO/<run-signature>/`. Existing runs
are not overwritten. Dry-run creates one test archive but never writes to HF.
Re-running resumes pending uploads before processing more candidates.

In [ ]:
# cell: archives
def dataset_readme():
    thresholds = {key: CONFIG[key] for key in (
        "NDVI_DROP_THRESHOLD", "MAX_LOSS_FRACTION", "MAX_LOSS_PATCH_AREA_M2",
        "MIN_VALID_FRACTION", "MEAN_NDVI_DROP_THRESHOLD", "REFLECTANCE_MIN",
        "REFLECTANCE_MAX",
    )}
    return (
        "# Multi-source landslide-candidate imagery\n\n"
        "Each ZIP contains `{incident_id}_{candidate_id}/` folders with "
        "`{id}_{cand}_{planet_before,planet_after,slope,aspect,s1_before,s1_after}.tif` "
        "and `provenance.json`, plus matching `metadata.csv` and `status.csv`.\n\n"
        f"Run: `{RUN_SIGNATURE}`. Archive prefix: `{remote_prefix()}`.\n\n"
        "Planet files are native Global Quarterly products resampled onto the output grid, "
        "including byte visual RGB/RGBA products. All delivered bands/sample values are "
        "retained in float32 storage. Band order, label source, datatype, native resolution, "
        "mosaic names and acquisition spans are recorded. Alpha is not NIR. "
        "Planet imagery is never used for NDVI.\n\n"
        "For screened candidates, `s2_screen_before.tif` and `s2_screen_after.tif` "
        "(with candidate prefix) retain separate Sentinel-2 B4/B8 composites as named "
        "red,nir float32 reflectance bands. They are auxiliary screening rasters, NOT "
        "the six stacked inputs: their default grid is 10 m over the same UTM footprint. "
        "Cloud/snow/shadow/nodata pixels are excluded using configured SCL classes before "
        "a per-band reflectance median. Source scene IDs, counts, windows and screening "
        "grid are recorded. Native B4/B8 resolution is 10 m; SCL is 20 m. "
        "Sentinel-1 bands are VV,VH in dB (already dB in GEE; not logged twice). "
        "Slope and aspect are degrees from the configured DEM.\n\n"
        "Candidates are processed in bounded thread-pool waves (MAX_WORKERS, default 4). "
        "Each worker has an isolated Planet HTTP session and writes only its candidate. "
        "Checkpoint/CSV writes, ZIP creation and uploads run on the coordinator thread; "
        "archives are built only between waves. Concurrency is reduced by batch slots "
        "and available disk headroom, reserving DOWNLOAD_WORKSPACE_GB per worker. "
        "This reserve is an estimate, not a guarantee of upstream quad sizes.\n\n"
        f"Footprints are {CONFIG['BOX_SIZE_M']} m squares in each candidate's local UTM CRS. "
        f"The requested pixel size is {CONFIG['OUTPUT_PIXEL_SIZE_M']} m; the actual pixel "
        "size is footprint size divided by the rounded pixel count. All output shapes match; "
        "CRSs and origins vary between candidates. Nearest-neighbour resampling does not "
        "increase the native resolution of Planet, SAR or DEM data.\n\n"
        f"Dry-season months: {CONFIG['DRY_SEASON_MONTHS']}. PRE ends strictly before "
        "the incident date; POST starts strictly after it. API window ends are exclusive. "
        "Only the exact Global Quarterly period is accepted; missing quads/mosaics are logged. "
        "Sentinel-2 screening uses the same PRE/POST windows, not the Planet pixel dates.\n\n"
        f"Year-shift fraction: {CONFIG['YEAR_SHIFT_FRACTION']}, target year: "
        f"{CONFIG['TARGET_YEAR']}, seed: {CONFIG['RANDOM_SEED']}, label restriction: "
        f"{CONFIG['YEAR_SHIFT_ONLY_LABEL']!r}. Original dates and labels are preserved. "
        "`year_shifted` records an actual changed date; `date_used` identifies original "
        "versus shifted imagery. Effective dates/windows describe the attempted final context "
        "even for incomplete candidates; check `status` before using them.\n\n"
        f"NDVI screening source: {CONFIG['S2_COLLECTION']}, requested screening pixel size "
        f"{CONFIG['S2_PIXEL_SIZE_M']} m, SCL clear classes {CONFIG['S2_CLEAR_SCL_CLASSES']}. "
        "Shifted pairs undergo NDVI-loss screening using jointly valid Sentinel-2 pixels and "
        "8-connected loss patches. Missing/insufficient data causes original-date fallback. "
        "Original-date statistics are logged but do not reject the original context. "
        "Missing/invalid Sentinel-2 data is inconclusive, never an NDVI pass. "
        "An original-date candidate can be complete for the six inputs even if screening "
        "is unavailable; inspect original_ndvi and screening_layers. "
        "The 2017-2018 surface-reflectance archive is not global; no TOA fallback is used. "
        "Fallback publishes only after all six replacement layers validate; terrain is reused.\n\n"
        "Thresholds (starting values, not universally validated):\n\n```json\n"
        + json.dumps(thresholds, indent=2) + "\n```\n\n"
        "Sentinel-2 SCL masking is imperfect; valid-fraction checks apply to both dates. "
        "NDVI changes are not landslide-specific; passing does not certify no landslide. "
        "Registration errors, seasonality and agriculture can affect screening.\n\n"
        "## Licensing and access\n\n"
        "Planet imagery may be restricted by your Planet contract. Private HF storage "
        "is the default, not permission to redistribute imagery. Confirm your Planet "
        "terms before uploading, including to a private third-party host. "
        "Observe the applicable Sentinel and DEM attribution/licensing terms. "
        "No blanket license for source imagery is granted by this generated dataset.\n"
    )


def remote_prefix():
    return CONFIG["HF_PATH_IN_REPO"].strip("/") + "/" + RUN_SIGNATURE[:16]


def ensure_hf_repo():
    retry_call(lambda: HF.create_repo(
        CONFIG["HF_REPO_ID"], repo_type="dataset", private=CONFIG["HF_PRIVATE"], exist_ok=True,
    ), "Create dataset repository")
    info = retry_call(lambda: HF.repo_info(CONFIG["HF_REPO_ID"], repo_type="dataset"), "Check repository")
    if CONFIG["HF_PRIVATE"] and not info.private:
        raise ValueError("Existing HF repository is public; use a private repository or explicitly "
                         "set HF_PRIVATE=False after verifying redistribution rights")


def folder_bytes(key):
    folder = ROOT / "candidates" / key
    return sum(path.stat().st_size for path in folder.iterdir() if path.is_file())


def verify_archive(path, keys):
    expected = set(keys)
    with zipfile.ZipFile(path) as archive:
        bad = archive.testzip()
        if bad:
            raise ValueError(f"ZIP CRC check failed for {bad}")
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("Archive contains duplicate file entries")
        folders = {name.split("/")[0] for name in names if "/" in name}
        if folders != expected:
            raise ValueError("Archive candidate folders do not match its completed-candidate list")
        with archive.open("status.csv") as handle:
            status = pd.read_csv(handle, dtype={"key": str})
        completed = set(status.loc[status.status.eq("complete"), "key"])
        if completed != expected or len(status) != len(expected):
            raise ValueError("Archive status.csv does not match complete candidate folders")
        for key in expected:
            required = {f"{key}/{key}_{layer}.tif" for layer in LAYERS}
            required.add(f"{key}/provenance.json")
            with archive.open(f"{key}/provenance.json") as handle:
                provenance = json.load(handle)
            for layer, status in provenance.get("screening_layers", {}).items():
                if status == "ok":
                    required.add(f"{key}/{key}_{layer}.tif")
            if not required.issubset(names):
                raise ValueError(f"Incomplete candidate in ZIP: {key}")
    print(f"Verified {Path(path).name}: {len(expected)} complete candidates")


def create_archive(keys):
    number = len(STATE["archives"]) + 1
    name = f"landslide_dataset_part{number:04d}.zip"
    destination = ROOT / "archives" / name
    partial = destination.with_suffix(".zip.tmp")
    records = [{**STATE["records"][key], "archive": name} for key in keys]
    for record in records:
        if record["status"] != "complete":
            raise ValueError("Only complete candidates may be archived")
        validate_folder(ROOT / "candidates" / record["key"], record)
    estimate = sum(folder_bytes(key) for key in keys)
    require_space(ROOT, math.ceil(estimate * 1.02) + 1024 ** 2)
    try:
        with tempfile.TemporaryDirectory(dir=ROOT / ".scratch", prefix="manifest_") as temporary:
            save_tables(records, temporary)
            (Path(temporary) / "README.md").write_text(dataset_readme())
            files = [(Path(temporary) / name, name) for name in ("metadata.csv", "status.csv", "README.md")]
            for key in keys:
                files.extend((path, f"{key}/{path.name}") for path in
                             sorted((ROOT / "candidates" / key).iterdir()) if path.is_file())
            with zipfile.ZipFile(partial, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
                for path, relative in tqdm(files, desc=f"ZIP batch {number}", leave=False):
                    archive.write(path, relative)
        verify_archive(partial, keys)
        os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)
    entry = {
        "name": name, "keys": list(keys), "size_bytes": destination.stat().st_size,
        "uploaded": False, "local_cleaned": False, "commit": None,
    }
    STATE["archives"].append(entry)
    for record in records:
        STATE["records"][record["key"]] = record
    checkpoint()
    return entry


def upload_archive(entry):
    path = ROOT / "archives" / entry["name"]
    if not entry["uploaded"]:
        verify_archive(path, entry["keys"])
        checkpoint()
        (ROOT / "README.md").write_text(dataset_readme())
        operations = [
            CommitOperationAdd(path_in_repo=f"{remote_prefix()}/{entry['name']}", path_or_fileobj=str(path)),
        ]
        for name in ("metadata.csv", "status.csv", "README.md", "invalid_rows.csv"):
            operations.append(CommitOperationAdd(
                path_in_repo=f"{remote_prefix()}/{name}", path_or_fileobj=str(ROOT / name),
            ))
        commit = retry_call(lambda: HF.create_commit(
            repo_id=CONFIG["HF_REPO_ID"], repo_type="dataset", operations=operations,
            commit_message=f"Upload verified candidate batch {entry['name']}",
        ), "Upload candidate batch")
        remote = retry_call(lambda: HF.get_paths_info(
            CONFIG["HF_REPO_ID"], paths=[f"{remote_prefix()}/{entry['name']}"],
            repo_type="dataset", revision=commit.oid,
        ), "Verify committed archive")
        if len(remote) != 1 or remote[0].size != entry["size_bytes"]:
            raise RuntimeError("Uploaded archive size verification failed; local files retained")
        entry["uploaded"], entry["commit"] = True, commit.oid
        checkpoint()
    if CONFIG["DELETE_AFTER_VERIFIED_UPLOAD"] and not entry["local_cleaned"]:
        for key in entry["keys"]:
            remove_child_tree(ROOT / "candidates" / key, ROOT / "candidates")
        path.unlink(missing_ok=True)
        entry["local_cleaned"] = True
        checkpoint()


def flush_batch(keys):
    if not keys:
        return
    available = disk_free(ROOT) - CONFIG["MIN_FREE_GB"] * 1024 ** 3
    estimate = sum(folder_bytes(key) for key in keys) * 1.02 + 1024 ** 2
    if estimate > available and len(keys) > 1:
        midpoint = len(keys) // 2
        flush_batch(keys[:midpoint])
        flush_batch(keys[midpoint:])
        return
    entry = create_archive(keys)
    if not CONFIG["DRY_RUN"]:
        upload_archive(entry)


def upload_final_manifests():
    (ROOT / "README.md").write_text(dataset_readme())
    names = ["metadata.csv", "status.csv", "README.md", "invalid_rows.csv"]
    operations = [CommitOperationAdd(
        path_in_repo=f"{remote_prefix()}/{name}", path_or_fileobj=str(ROOT / name),
    ) for name in names]
    # Do not overwrite an existing dataset card, which may contain its own terms.
    existing = retry_call(lambda: HF.list_repo_files(
        CONFIG["HF_REPO_ID"], repo_type="dataset",
    ), "List repository files")
    if "README.md" not in existing:
        operations.append(CommitOperationAdd(path_in_repo="README.md", path_or_fileobj=str(ROOT / "README.md")))
    retry_call(lambda: HF.create_commit(
        CONFIG["HF_REPO_ID"], repo_type="dataset", operations=operations,
        commit_message="Update multi-source candidate manifests",
    ), "Upload final manifests")


def run_dataset():
    if not CONFIG["DRY_RUN"]:
        ensure_hf_repo()
    for entry in STATE["archives"]:
        if not CONFIG["DRY_RUN"]:
            upload_archive(entry)
        elif not entry["uploaded"]:
            verify_archive(ROOT / "archives" / entry["name"], entry["keys"])
    pending = []
    keys = [key for key, record in STATE["records"].items() if not record.get("archive")]
    cursor = 0
    with ThreadPoolExecutor(max_workers=CONFIG["MAX_WORKERS"]) as executor, tqdm(
        total=len(keys), desc="Candidate pairs",
    ) as progress:
        while cursor < len(keys):
            size = sum(folder_bytes(key) for key in pending)
            reserve = CONFIG["MIN_FREE_GB"] * 1024 ** 3
            workspace = CONFIG["DOWNLOAD_WORKSPACE_GB"] * 1024 ** 3
            # Retain headroom for a ZIP alongside completed folders, plus one
            # download workspace per admitted worker. No unbounded task queue.
            archive_headroom = math.ceil(size * 1.02) + 1024 ** 2 if pending else 0
            available = disk_free(ROOT) - reserve - archive_headroom
            capacity = (max(0, math.floor(available / workspace)) if workspace
                        else CONFIG["MAX_WORKERS"] if available >= 0 else 0)
            if pending and (
                len(pending) >= CONFIG["BATCH_SIZE"]
                or size >= CONFIG["MAX_BATCH_UNCOMPRESSED_GB"] * 1024 ** 3
                or capacity < 1
            ):
                flush_batch(pending)
                pending = []
                continue
            if capacity < 1:
                raise OSError("Insufficient disk headroom for one candidate worker; "
                              "free space or reduce DOWNLOAD_WORKSPACE_GB after checking quad sizes")
            wave_size = min(CONFIG["MAX_WORKERS"], capacity, len(keys) - cursor,
                            CONFIG["BATCH_SIZE"] - len(pending))
            wave = keys[cursor:cursor + wave_size]
            cursor += wave_size
            futures = {executor.submit(process_candidate_worker, STATE["records"][key]): key for key in wave}
            for future in as_completed(futures):
                key = futures[future]
                try:
                    record = future.result()
                except Exception as exc:
                    LOG.error("%s processing failed: %s", key, error_summary(exc))
                    record = deepcopy(STATE["records"][key])
                    record["status"] = "failed"
                    record["processing_error"] = error_summary(exc)
                STATE["records"][key] = record
                checkpoint()
                if record["status"] == "complete":
                    pending.append(key)
                elif (ROOT / ".staging" / key).exists():
                    LOG.warning("%s incomplete; staging retained at %s", key, ROOT / ".staging" / key)
                progress.update(1)
            # Flush only after every worker in the wave has stopped writing.
            size = sum(folder_bytes(key) for key in pending)
            print(f"Disk free: {disk_free(ROOT) / 1024 ** 3:.2f} GiB")
            if (len(pending) >= CONFIG["BATCH_SIZE"]
                    or size >= CONFIG["MAX_BATCH_UNCOMPRESSED_GB"] * 1024 ** 3):
                flush_batch(pending)
                pending = []
    flush_batch(pending)
    checkpoint()
    (ROOT / "README.md").write_text(dataset_readme())
    if not CONFIG["DRY_RUN"]:
        upload_final_manifests()
    report_run()


def report_run():
    records = list(STATE["records"].values())
    table = pd.DataFrame([flatten_record(record) for record in records])
    print("Candidate statuses:")
    print(table.status.value_counts().to_string())
    shifted = table.loc[table.year_shifted]
    print(f"Actually shifted dates: {len(shifted)}")
    if len(shifted):
        print("Shifted screening outcomes:")
        print(shifted.ndvi_check_result.value_counts(dropna=False).to_string())
        fallbacks = (shifted.loc[shifted.fallback_triggered.eq(True)]
                     if "fallback_triggered" in shifted else shifted.iloc[:0])
        print(f"Fallbacks triggered: {len(fallbacks)}")
        if len(fallbacks):
            print(fallbacks.ndvi_check_reason.value_counts().to_string())
    complete = table.loc[table.status.eq("complete")]
    years = pd.to_datetime(complete.effective_incident_date).dt.year
    print(f"Complete candidates: {len(complete)}/{len(table)}")
    for name, mask in (
        ("2017-2019", years.between(2017, 2019)),
        ("2024-2026", years.between(2024, 2026)),
        ("other", ~years.between(2017, 2019) & ~years.between(2024, 2026)),
    ):
        fraction = float(mask.mean()) if len(complete) else 0
        print(f"Final complete-candidate era {name}: {int(mask.sum())} ({fraction:.1%})")
    if CONFIG["DRY_RUN"]:
        print("DRY RUN: no HF writes. Test archive(s):", ROOT / "archives")
        if not STATE["archives"]:
            raise RuntimeError("Dry-run did not produce a complete candidate/test ZIP; inspect status.csv "
                               "and resolve missing mosaic/band/access issues before a full run")
        if len(complete) != CONFIG["DRY_RUN_COUNT"]:
            raise RuntimeError("Dry-run is partial: not all three candidates completed; inspect status.csv")
    else:
        print(f"https://huggingface.co/datasets/{CONFIG['HF_REPO_ID']}/tree/main/{remote_prefix()}")

## 9. Authenticate, discover the catalogue, and initialize the run
The series catalogue prints names and IDs only. Global Quarterly is selected
by exact name. Remove an old analytic monthly series ID from CONFIG when
upgrading. An exact `global_quarterly_{year}q{quarter}_mosaic` template is used.
A changed data configuration requires a fresh OUTPUT_DIR to avoid stale imagery;
MAX_WORKERS alone may be adjusted when resuming.

In [ ]:
# cell: initialize
ROOT, ALL_ROWS, STATE, RUN_SIGNATURE = prepare_run()
checkpoint()
PLANET, HF, AVAILABLE_SERIES = authenticate()
PLANET_SERIES = choose_planet_series(AVAILABLE_SERIES)
series_mosaics_url = (
    PLANET_SERIES.get("_links", {}).get("mosaics")
    or BASEMAPS_URL + f"series/{PLANET_SERIES['id']}/mosaics/"
)
MOSAIC_CATALOGUE = list(planet_pages(PLANET, series_mosaics_url, "mosaics"))
print("Selected Planet series:", PLANET_SERIES.get("name"))
print("Accessible mosaics:", len(MOSAIC_CATALOGUE))
print("Example mosaic names:", [item.get("name") for item in MOSAIC_CATALOGUE[:8]])
if not MOSAIC_CATALOGUE:
    raise RuntimeError("Selected series has no accessible mosaics; check entitlement/time range")
if not CONFIG["PLANET_MOSAIC_NAME_TEMPLATE"] and not any(
    nominal_mosaic_window(mosaic) for mosaic in MOSAIC_CATALOGUE
):
    raise ValueError("Catalogue does not encode YYYYqQ; set PLANET_MOSAIC_NAME_TEMPLATE "
                     "from the printed names using {year} and {quarter}")
report_quarterly_coverage(list(STATE["records"].values()), MOSAIC_CATALOGUE)
print("NDVI source:", CONFIG["S2_COLLECTION"], "(B4/B8); Planet imagery is not used for NDVI")
LOG.warning("Sentinel-2 SR coverage in 2017-2018 is not global; missing/insufficient "
            "screening pixels cause original-date fallback, never a false pass")
print("Candidates selected:", len(STATE["records"]))
print("Actually shifted in selected rows:",
      sum(record["year_shifted"] for record in STATE["records"].values()))
example = next(iter(STATE["records"].values()))["grid"]
print("Output shape:", example["height"], example["width"],
      "actual pixel size (m):", example["pixel_size_m"])
if not CONFIG["DRY_RUN"]:
    ensure_hf_repo()

## 10. Optional calibration (before production)
A negative-only input cannot calibrate landslide sensitivity. Attach a labeled
positive CSV using CALIBRATION_LANDSLIDE_CSV. Calibration samples known
2017–2019 positives and shifted negatives, prints statistics and plots, then
deliberately stops. Review thresholds and start a fresh run directory.

In [ ]:
# cell: calibrate
calibrate_ndvi(ALL_ROWS)

## 11. Process, screen, fall back, archive, and upload
Set MAX_WORKERS (default 4) to control concurrent candidate pairs; use 1 for
sequential operation. Downloads/layers within a candidate remain sequential.
Work is submitted in bounded waves; no next wave starts until the current
wave finishes. Metadata and HF uploads are serialized. Per-worker HTTP sessions
are closed even after failures. Disk headroom may reduce actual concurrency.
Keep DOWNLOAD_WORKSPACE_GB large enough for a full Planet quad per worker and
intermediate files. Existing byte/chunk disk checks remain in force.
#
Dry-run processes exactly three selected candidates including a shifted one.
It still authenticates/downloads and performs screening/fallback, but does not
create or write an HF repository. Successful production batches are verified
remotely before optional local cleanup. Save Kaggle outputs to retain resume
state between sessions; a fresh Kaggle filesystem cannot recover local state.

In [ ]:
# cell: run
run_dataset()